# Explore one TikTok end-to-end

This deliberately small notebook downloads one public TikTok and persists both the MP4 and its public metadata. It tries **Pyktok first when browser cookies/DBus are available**, then reports its error before using the current yt-dlp release as a narrow fallback. On headless Linux/Jupyter, Pyktok is skipped explicitly because browser-cookie3 cannot read cookies without DBus. TikTok changes frequently, so failures include their original exception and are never ignored.

Setup (Python 3.10+ recommended): run the next cell once. In Google Colab, do not set `PYKTOK_FORCE`: Colab has no local browser profile/DBus, so the notebook uses yt-dlp and then the public webpage JSON fallback. Pyktok may need cookies from a locally installed Chrome/Firefox profile; set `BROWSER` below only on such a local runtime. Only download content you are entitled to collect and follow TikTok's terms and applicable law.

In [ ]:
%pip install -q --upgrade "pyktok==0.0.31" "yt-dlp>=2026.07.04"


In [ ]:
import json
import os
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, display

# Public example from Pyktok's own documentation. Replace this one value to explore another video.
TIKTOK_URL = "https://www.tiktok.com/@tiktok/video/7106594312292453675"
BROWSER = os.environ.get("PYKTOK_BROWSER", "chrome")  # e.g. chrome or firefox
# In headless Linux/Jupyter environments browser-cookie3 can require DBus just to read cookies.
# Set PYKTOK_FORCE=1 only when you have configured browser cookies and want to force Pyktok.
PYKTOK_FORCE = os.environ.get("PYKTOK_FORCE", "0") == "1"

match = re.search(r"/video/(\d+)", TIKTOK_URL)
if not match:
    raise ValueError(f"Expected a canonical TikTok URL containing /video/<id>: {TIKTOK_URL}")
video_id = match.group(1)
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = repo_root / "data" / "exploration" / video_id
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / "video.mp4"
metadata_path = output_dir / "metadata.json"
print(f"Source: {TIKTOK_URL}\nOutput: {output_dir.resolve()}")


## Download and collect metadata

Pyktok writes the MP4 beside its CSV, so the cell runs it inside the final exploration directory and then gives the file the stable name `video.mp4`. If Pyktok fails, the full error is printed before `yt-dlp` is attempted. If both fail, the cell raises a combined, actionable error.

In [ ]:
def json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)

def first_present(data, *keys):
    lowered = {str(k).lower(): v for k, v in data.items()}
    for key in keys:
        value = lowered.get(key.lower())
        if value is not None and str(value) not in {"", "nan", "None"}:
            return json_safe(value)
    return None

def publication_value(raw):
    value = first_present(raw, "video_timestamp", "create_time", "timestamp", "upload_date")
    if isinstance(value, (int, float)) and value > 10**9:
        return datetime.fromtimestamp(value, tz=timezone.utc).isoformat()
    if isinstance(value, str) and value.isdigit() and int(value) > 10**9:
        return datetime.fromtimestamp(int(value), tz=timezone.utc).isoformat()
    if isinstance(value, str) and len(value) == 8 and value.isdigit():
        return f"{value[:4]}-{value[4:6]}-{value[6:]}"
    return value

def normalize_metadata(raw, collector):
    caption = first_present(raw, "video_description", "description", "desc", "title")
    hashtags = first_present(raw, "video_hashtags", "hashtags", "tags")
    if isinstance(hashtags, str):
        hashtags = re.findall(r"#([\w.-]+)", hashtags) or [x.strip() for x in hashtags.split(",") if x.strip()]
    if not hashtags and caption:
        hashtags = re.findall(r"#([\w.-]+)", caption)
    return {
        "source_url": TIKTOK_URL,
        "video_id": video_id,
        "collector": collector,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "creator": first_present(raw, "author_name", "author", "uploader", "creator", "author_uniqueid"),
        "caption": caption,
        "hashtags": hashtags or [],
        "views": first_present(raw, "video_playcount", "playcount", "view_count"),
        "likes": first_present(raw, "video_diggcount", "diggcount", "like_count"),
        "comments": first_present(raw, "video_commentcount", "commentcount", "comment_count"),
        "shares": first_present(raw, "video_sharecount", "sharecount", "repost_count"),
        "duration_seconds": first_present(raw, "video_duration", "duration"),
        "publication_date": publication_value(raw),
        "raw": {str(k): json_safe(v) for k, v in raw.items()},
    }

def collect_with_pyktok():
    import pandas as pd
    import pyktok as pyk
    csv_path = output_dir / "pyktok_metadata.csv"
    before = set(output_dir.glob("*.mp4"))
    previous_cwd = Path.cwd()
    try:
        os.chdir(output_dir)
        pyk.specify_browser(BROWSER)
        pyk.save_tiktok(TIKTOK_URL, True, csv_path.name, BROWSER)
    finally:
        os.chdir(previous_cwd)
    candidates = [p for p in output_dir.glob("*.mp4") if p not in before]
    if not candidates and video_path.exists():
        candidates = [video_path]
    if not candidates:
        raise FileNotFoundError(f"Pyktok returned without creating an MP4 in {output_dir}")
    downloaded = max(candidates, key=lambda p: p.stat().st_mtime)
    if downloaded != video_path:
        downloaded.replace(video_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Pyktok returned without creating metadata CSV: {csv_path}")
    rows = pd.read_csv(csv_path).to_dict(orient="records")
    if not rows:
        raise ValueError(f"Pyktok metadata CSV is empty: {csv_path}")
    return normalize_metadata(rows[-1], "pyktok")

def collect_with_ytdlp():
    from yt_dlp import YoutubeDL
    options = {
        "outtmpl": str(video_path),
        "format": "best[ext=mp4]/best",
        "noplaylist": True,
        "quiet": False,
    }
    with YoutubeDL(options) as ydl:
        info = ydl.extract_info(TIKTOK_URL, download=True)
    return normalize_metadata(info, "yt-dlp (explicit fallback after Pyktok failure)")

def find_video_item(value):
    """Find TikTok's public itemStruct in the universal page JSON."""
    if isinstance(value, dict):
        video = value.get("video")
        if isinstance(video, dict) and (video.get("downloadAddr") or video.get("playAddr")):
            return value
        for child in value.values():
            found = find_video_item(child)
            if found is not None:
                return found
    elif isinstance(value, list):
        for child in value:
            found = find_video_item(child)
            if found is not None:
                return found
    return None

def collect_with_webpage():
    """Fallback for Colab/IPs where yt-dlp's TikTok extractor is blocked."""
    import requests
    from html import unescape
    page_headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/131 Safari/537.36"}
    session = requests.Session()
    page = session.get(TIKTOK_URL, headers=page_headers, timeout=30)
    page.raise_for_status()
    script_match = re.search(
        r'<script id="__UNIVERSAL_DATA_FOR_REHYDRATION__"[^>]*>(.*?)</script>',
        page.text,
        flags=re.DOTALL,
    )
    if not script_match:
        raise ValueError("TikTok page did not contain __UNIVERSAL_DATA_FOR_REHYDRATION__")
    document = json.loads(unescape(script_match.group(1)))
    item = find_video_item(document)
    if item is None:
        raise ValueError("TikTok page JSON did not contain a downloadable video item")
    video = item["video"]
    download_url = video.get("downloadAddr") or video.get("playAddr")
    download_headers = {**page_headers, "Referer": TIKTOK_URL}
    # Keep the cookies set by the page request; TikTok's CDN may reject a bare URL.
    with session.get(download_url, headers=download_headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with video_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    author = item.get("author") or {}
    stats = item.get("stats") or item.get("statsV2") or {}
    challenges = item.get("challenges") or []
    raw = {
        "author_name": author.get("uniqueId") or author.get("nickname"),
        "video_description": item.get("desc"),
        "video_hashtags": [tag.get("title") for tag in challenges if isinstance(tag, dict)],
        "video_playcount": stats.get("playCount"),
        "video_diggcount": stats.get("diggCount"),
        "video_commentcount": stats.get("commentCount"),
        "video_sharecount": stats.get("shareCount"),
        "video_duration": video.get("duration"),
        "create_time": item.get("createTime"),
        "webpage_item": item,
    }
    return normalize_metadata(raw, "TikTok webpage JSON (after yt-dlp failure)")

pyktok_error = None
metadata = None
if not PYKTOK_FORCE and not os.environ.get("DBUS_SESSION_BUS_ADDRESS"):
    pyktok_error = (
        "Pyktok skipped: DBUS_SESSION_BUS_ADDRESS is not set, so browser-cookie3 cannot read cookies in this headless environment. Set PYKTOK_FORCE=1 after configuring browser cookies to retry it."
    )
    print(pyktok_error)
else:
    try:
        metadata = collect_with_pyktok()
    except Exception as exc:
        pyktok_error = f"{type(exc).__name__}: {exc}"
        print(f"Pyktok failed explicitly: {pyktok_error}")

if metadata is None:
    print("Trying the documented yt-dlp fallback...")
    try:
        metadata = collect_with_ytdlp()
        if pyktok_error:
            metadata["pyktok_error"] = pyktok_error
    except Exception as ytdlp_exc:
        ytdlp_error = f"{type(ytdlp_exc).__name__}: {ytdlp_exc}"
        print(f"yt-dlp failed explicitly: {ytdlp_error}")
        print("Trying the public TikTok webpage JSON fallback...")
        try:
            metadata = collect_with_webpage()
            metadata["yt_dlp_error"] = ytdlp_error
            if pyktok_error:
                metadata["pyktok_error"] = pyktok_error
        except Exception as webpage_exc:
            raise RuntimeError(
                "All collectors failed. In Colab, rerun the install cell and verify the URL is public. "
                f"Pyktok: {pyktok_error or 'not attempted'}; yt-dlp: {ytdlp_error}; "
                f"webpage JSON: {type(webpage_exc).__name__}: {webpage_exc}"
            ) from webpage_exc

if not video_path.exists() or video_path.stat().st_size == 0:
    raise RuntimeError(f"Collector completed but MP4 is missing or empty: {video_path}")
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {video_path} ({video_path.stat().st_size / 1_048_576:.1f} MiB)")
print(f"Saved {metadata_path}")


## Manual verification

Compare the source URL/video ID, creator and caption below with the embedded MP4. The complete collector response remains under `raw` in `metadata.json` for debugging.

In [ ]:
summary_fields = ["source_url", "video_id", "collector", "creator", "caption", "hashtags", "views", "likes", "comments", "shares", "duration_seconds", "publication_date"]
print(json.dumps({key: metadata.get(key) for key in summary_fields}, ensure_ascii=False, indent=2))
video_b64 = __import__("base64").b64encode(video_path.read_bytes()).decode("ascii")
display(HTML(f'<video controls style="max-width: 480px"><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>'))


In [ ]:
# Final persisted-artifact checks (fail loudly if the run is incomplete).
persisted = json.loads(metadata_path.read_text(encoding="utf-8"))
assert persisted["source_url"] == TIKTOK_URL
assert persisted["video_id"] == video_id
assert video_path.stat().st_size > 0
print("Verification passed:", video_path.resolve(), metadata_path.resolve())
